In [ ]:
import sys

import numpy as np
import PIL.Image as Image
import scipy.io as sio
import tifffile
import torch

sys.path.append("/Data4/cao/ZiHanCao/exps/HyperspectralTokenizer")

from src.stage2.segmentation.data.robust_sampler import RobustHyperspectralSampler, sample_img_with_gt_indices
from src.utilities.io import read_image
from src.utilities.train_utils.visualization import visualize_segmentation_map

In [ ]:
img = read_image(
    "/Data4/cao/ZiHanCao/exps/HyperspectralTokenizer/data/Downstreams/ClassificationCollection/cls/mat/PaviaU.mat",
    dict_extract_type="last",
)
gt = read_image(
    "/Data4/cao/ZiHanCao/exps/HyperspectralTokenizer/data/Downstreams/ClassificationCollection/cls/cls_GT/PaviaU_gt.mat",
)
gt2 = read_image(
    "/Data4/cao/ZiHanCao/exps/HyperspectralTokenizer/data/Downstreams/ClassificationCollection/cls/gt_maps/PaviaU_gt_map.png"
)

img.shape, gt.shape, gt2.shape

In [ ]:
# plot the image
def show_img(img: np.ndarray, slices: tuple[slice, slice] | None = None, chans=None):
    """show image"""
    img = img[*slices, :] if slices is not None else img
    img_min, img_max = img.min(), img.max()
    img = (img - img_min) / (img_max - img_min)

    # mean of the image bands
    n = img.shape[-1] // 3
    if chans is None:
        img_rgb = np.zeros((img.shape[0], img.shape[1], 3))
        for i in range(3):
            img_rgb[:, :, i] = img[:, :, i * n : (i + 1) * n].mean(axis=-1)
    else:
        img_rgb = img[:, :, chans]
    return (img_rgb * 255.0).astype(np.uint8)


slices = (slice(None), slice(None))
img_rgb = show_img(img, slices, [39, 29, 19])
gt_rgb = show_img(gt2, slices)
img_cat = np.concatenate([img_rgb, gt_rgb], axis=1)
Image.fromarray(img_cat)

In [ ]:
shape_xy = tuple(img.shape[:2])
mesh = np.concatenate(np.meshgrid(*[np.arange(i) for i in shape_xy]))
# print(mesh.shape)
mesh_flat = mesh.flatten()

assert gt.ndim == 2
gt_flat = gt.flatten()
classes = np.unique(gt_flat)
print(f"Has classes: {classes}")
n_cls = classes.max() + 1
print(n_cls)

# sample
# 0 is the background
indices_per_class = []
for c in range(n_cls - 1):
    indices_c = np.where(gt_flat == c)[0]
    indices_per_class.append(indices_c)
    print(f"Class {c} has {len(indices_c)} samples")

In [ ]:
import random

ps = 64  # patch size
ratio = 0.2
all_n_pixels = shape_xy[0] * shape_xy[1]
n_pixels_select = int(ratio * all_n_pixels)
n_pix_per_cls = int(n_pixels_select / n_cls)
print(f"Select {n_pixels_select} pixels in total")

# random select indices
indices_pc_select = []

for c in range(n_cls - 1):
    random.sample

In [ ]:
sampler = RobustHyperspectralSampler(
    gt_2d=gt,
    balance_strategy="equal_size",
    random_seed=2025,
    verbose=True,
)

sampled_indices = sampler.sample(samples_per_class=5)
patches_dict, labels_dict, unsampled_area = sample_img_with_gt_indices(img, gt, sampled_indices, patch_size=64)

In [ ]:
p1, label1 = patches_dict[3], labels_dict[3]
p1_1, label1_1 = p1[1], label1[1]


p1_1 = show_img(p1_1, chans=[39, 29, 19])
label1_1 = visualize_segmentation_map(label1_1, n_class=10)

import matplotlib.pyplot as plt

plt.subplot(1, 3, 1)
plt.imshow(p1_1)
plt.subplot(1, 3, 2)
plt.imshow(label1_1)
plt.subplot(1, 3, 3)
plt.imshow(unsampled_area)